# UWF-ZeekData24 Dataset Cleaning

This notebook processes and cleans the Zeek network flow dataset for thesis analysis.

**Cleaning Pipeline Order:**
1. Install & Import Libraries
2. Load Dataset
3. Initial Exploration
4. Type Conversion 
5. Handle Missing Values
6. Remove Duplicates
7. IP Address Validation
8. Remove Outliers
9. Standardize Columns
10. Clean label_binary
11. Drop Useless Columns
12. Data Quality Checks
13. Label Distribution
14. Statistical Summary
15. Save Cleaned Dataset

## 1. Install & Import Libraries

In [5]:
%pip install scikit-learn imbalanced-learn matplotlib seaborn pandas numpy openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\devMrX\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import os
import glob
import ipaddress
import logging
from pathlib import Path

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Pandas version: 3.0.1
NumPy version: 2.4.4


## 2. Load Dataset

In [7]:
# Define dataset directory
dataset_dir = r'E:\thesis\dataset\UWF-ZeekData24'

# Find all CSV files
csv_files = glob.glob(os.path.join(dataset_dir, '*.csv'))
print(f"Found {len(csv_files)} CSV files")

# Load and concatenate all CSV files
dfs = []
for file in csv_files:
    try:
        df_tmp = pd.read_csv(file, low_memory=False) 
        dfs.append(df_tmp)
        print(f"Loaded: {os.path.basename(file)} - Shape: {df_tmp.shape}")
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Combine all dataframes
df = pd.concat(dfs, ignore_index=True)
initial_shape = df.shape
print(f"\nCombined dataset shape: {initial_shape}")

Found 33 CSV files
Loaded: part-00000-0615e778-641c-4399-8d78-f14cb2da50bd-c000.csv - Shape: (66724, 27)
Loaded: part-00000-0af89d10-df53-44fd-b124-a8a496fd5023-c000.csv - Shape: (23003, 23)
Loaded: part-00000-0d012414-989a-4b0f-937d-36b02cacf398-c000.csv - Shape: (29, 27)
Loaded: part-00000-15e3dd03-ea76-429e-a52a-ce96a90517f9-c000.csv - Shape: (10490, 23)
Loaded: part-00000-20202de3-67ac-4628-90e5-6f7a8153737d-c000.csv - Shape: (56, 26)
Loaded: part-00000-27b9c76f-8291-49ef-bd5a-c47c18444b18-c000.csv - Shape: (28744, 26)
Loaded: part-00000-2ac3ee1a-f94a-44bb-9413-dbfa36b751da-c000.csv - Shape: (28964, 26)
Loaded: part-00000-30aee0c5-aea5-4666-aa1d-36370893753a-c000 (1).csv - Shape: (22860, 27)
Loaded: part-00000-30aee0c5-aea5-4666-aa1d-36370893753a-c000.csv - Shape: (22860, 27)
Loaded: part-00000-318611a1-7cdc-4dd0-9348-c6368917fd0c-c000.csv - Shape: (10260, 23)
Loaded: part-00000-4890c258-40e8-4ef8-b345-06d436539b95-c000.csv - Shape: (20, 26)
Loaded: part-00000-49ed1cc6-3205-4c76-8e

## 3. Initial Data Exploration

In [8]:
print("Column names and types:")
print(df.dtypes)
print(f"\nDataset shape: {df.shape}")
print(f"\nMissing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nFirst few rows:")
df.head()

Column names and types:
community_id                str
conn_state                  str
duration                float64
history                     str
src_ip_zeek                 str
src_port_zeek           float64
dest_ip_zeek                str
dest_port_zeek          float64
local_orig                 bool
local_resp                 bool
missed_bytes              int64
orig_bytes              float64
orig_ip_bytes             int64
orig_pkts                 int64
proto                       str
resp_bytes              float64
resp_ip_bytes             int64
resp_pkts                 int64
service                  object
ts                      float64
uid                         str
datetime                    str
vlan                    float64
label_tactic                str
label_technique             str
label_binary             object
label_cve                   str
protocol                    str
dest_ip                     str
dest_port               float64
src_port        

,community_id,conn_state,duration,history,src_ip_zeek,src_port_zeek,dest_ip_zeek,dest_port_zeek,local_orig,local_resp,...,label_tactic,label_technique,label_binary,label_cve,protocol,dest_ip,dest_port,src_port,src_ip,mitre_attack_tactics
0,1:GdRYleG142qEcmSwp4+jaKhgRTg=,S0,NaN,S,143.88.5.14,33440.0,143.88.5.12,23574.0,False,False,...,Discovery,T1046,True,none,NaN,NaN,NaN,NaN,NaN,NaN
1,1:JLDqMOybeshjgKBOe4CxvMRNKD4=,REJ,0.000077,Sr,143.88.5.14,40560.0,143.88.5.11,45377.0,False,False,...,Discovery,T1046,True,none,NaN,NaN,NaN,NaN,NaN,NaN
2,1:7wO/5ndh80L2CKVPVvWucl/Tbrw=,S0,NaN,S,143.88.11.11,48007.0,143.88.11.15,18512.0,False,False,...,Discovery,T1046,True,none,NaN,NaN,NaN,NaN,NaN,NaN
3,1:jCHDFrfm5ruXr2wCmIjKKJCGJOg=,S0,NaN,S,143.88.5.14,46142.0,143.88.5.13,42761.0,False,False,...,Discovery,T1046,True,none,NaN,NaN,NaN,NaN,NaN,NaN
4,1:t1xloEe/wUBJNPUO55aYQZE1PYA=,S0,NaN,S,143.88.11.11,50588.0,143.88.11.1,1444.0,False,False,...,Discovery,T1046,True,none,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Data Type Conversion


In [9]:
# Replace empty strings / 'None' strings with actual NaN
df = df.replace('', np.nan)
df = df.replace('None', np.nan)

# Convert datetime column
if 'datetime' in df.columns:
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce', utc=True)

# Convert ts (Unix timestamp)
if 'ts' in df.columns:
    df['ts'] = pd.to_numeric(df['ts'], errors='coerce')

# Convert numeric columns
numeric_cols = [
    'duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts',
    'orig_ip_bytes', 'resp_ip_bytes', 'src_port_zeek', 'dest_port_zeek',
    'missed_bytes', 'vlan'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert boolean columns
bool_cols = ['local_orig', 'local_resp']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})

print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
community_id                            str
conn_state                              str
duration                            float64
history                                 str
src_ip_zeek                             str
src_port_zeek                       float64
dest_ip_zeek                            str
dest_port_zeek                      float64
local_orig                             bool
local_resp                             bool
missed_bytes                          int64
orig_bytes                          float64
orig_ip_bytes                         int64
orig_pkts                             int64
proto                                   str
resp_bytes                          float64
resp_ip_bytes                         int64
resp_pkts                             int64
service                              object
ts                                  float64
uid                                     str
datetime                datetime64[us, UTC]
vla

## 5. Handle Missing Values

In [10]:
# --- 5a. Drop rows missing critical network identifiers ---
critical_cols = ['src_ip_zeek', 'dest_ip_zeek', 'proto']
missing_before = len(df)
df = df.dropna(subset=critical_cols)
print(f"Removed {missing_before - len(df)} rows with missing critical values")

# --- 5b. Fill numeric network stats with 0 ---
# (ในเชิงเครือข่ายหมายถึง 'ไม่มีการส่งข้อมูล')
numeric_fill_cols = [
    'duration', 'orig_bytes', 'resp_bytes',
    'orig_pkts', 'resp_pkts', 'orig_ip_bytes',
    'resp_ip_bytes', 'missed_bytes', 'vlan'
]
for col in numeric_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# --- 5c. Fill categorical / label columns ---
# ใช้ค่าเดียวกันทั้ง notebook (ไม่สลับระหว่าง 'none' กับ 'Unknown')
cat_fill_strategy = {
    'service':              'unknown',
    'label_tactic':         'none',
    'label_technique':      'none',
    'label_cve':            'none',
    'mitre_attack_tactics': 'none',
    'history':              '',
    'conn_state':           'unknown',
}
for col, fill_val in cat_fill_strategy.items():
    if col in df.columns:
        df[col] = df[col].fillna(fill_val)

print(f"\nRemaining missing values: {df.isnull().sum().sum()}")
remaining = df.isnull().sum()
if remaining.sum() > 0:
    print(remaining[remaining > 0])

Removed 2044734 rows with missing critical values

Remaining missing values: 1703795
protocol     340759
dest_ip      340759
dest_port    340759
src_port     340759
src_ip       340759
dtype: int64


## 6. Remove Duplicates

In [11]:
dupes_before = len(df)

# ใช้ uid เป็น unique identifier ของแต่ละ connection
# หาก uid ไม่มีหรือ null ให้ fallback เป็น drop_duplicates ทุก column
if 'uid' in df.columns and df['uid'].notna().all():
    df = df.drop_duplicates(subset=['uid'])
    print(f"Removed duplicates based on 'uid'")
elif 'community_id' in df.columns:
    df = df.drop_duplicates(subset=['community_id'])
    print(f"Removed duplicates based on 'community_id'")
else:
    df = df.drop_duplicates()
    print(f"Removed duplicates based on all columns")

print(f"Removed {dupes_before - len(df)} duplicate rows")
print(f"Dataset shape: {df.shape}")

Removed duplicates based on 'uid'
Removed 47856 duplicate rows
Dataset shape: (292903, 33)


## 7. IP Address Validation


In [12]:
import re

# Pre-compiled regex patterns (เร็วกว่า compile ทุก row)
_IPV4_RE = re.compile(
    r'^((25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(25[0-5]|2[0-4]\d|[01]?\d\d?)$'
)
_IPV6_RE = re.compile(
    r'^([0-9a-fA-F]{0,4}:){2,7}[0-9a-fA-F]{0,4}$'
)

def is_valid_ip_mask(series: pd.Series) -> pd.Series:
    """
    Return boolean mask (True = valid IP) แบบ vectorized
    รองรับทั้ง IPv4 และ IPv6
    """
    s = series.astype(str).str.strip()
    return (
        s.str.match(_IPV4_RE) |
        s.str.match(_IPV6_RE)
    ) & series.notna()

rows_before = len(df)

if 'src_ip_zeek' in df.columns:
    valid_src_mask = is_valid_ip_mask(df['src_ip_zeek'])
    invalid_src = df[~valid_src_mask]
    print(f"Invalid src_ip_zeek samples ({len(invalid_src)} rows):")
    print(invalid_src['src_ip_zeek'].value_counts().head(10))
    df = df[valid_src_mask]

if 'dest_ip_zeek' in df.columns:
    valid_dst_mask = is_valid_ip_mask(df['dest_ip_zeek'])
    invalid_dst = df[~valid_dst_mask]
    print(f"\nInvalid dest_ip_zeek samples ({len(invalid_dst)} rows):")
    print(invalid_dst['dest_ip_zeek'].value_counts().head(10))
    df = df[valid_dst_mask]

print(f"\nRemoved {rows_before - len(df)} rows with invalid IPs")
print(f"Dataset shape: {df.shape}")

Invalid src_ip_zeek samples (0 rows):
Series([], Name: count, dtype: int64)

Invalid dest_ip_zeek samples (0 rows):
Series([], Name: count, dtype: int64)

Removed 0 rows with invalid IPs
Dataset shape: (292903, 33)


## 8. Remove Outliers

In [13]:
rows_before = len(df)

# Remove rows with invalid port numbers (0–65535)
if 'src_port_zeek' in df.columns:
    df = df[(df['src_port_zeek'] >= 0) & (df['src_port_zeek'] <= 65535)]

if 'dest_port_zeek' in df.columns:
    df = df[(df['dest_port_zeek'] >= 0) & (df['dest_port_zeek'] <= 65535)]

# Remove rows with negative bytes/packets
for col in ['orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts', 'missed_bytes']:
    if col in df.columns:
        df = df[df[col] >= 0]

print(f"Removed {rows_before - len(df)} outlier rows")
print(f"Dataset shape: {df.shape}")

Removed 0 outlier rows
Dataset shape: (292903, 33)


## 9. Standardize Columns

In [14]:
# Lowercase protocol names
if 'proto' in df.columns:
    df['proto'] = df['proto'].str.lower().str.strip()

# Uppercase connection states
if 'conn_state' in df.columns:
    df['conn_state'] = df['conn_state'].str.upper().str.strip()

# Lowercase service names
if 'service' in df.columns:
    df['service'] = df['service'].str.lower().str.strip()

print("Columns standardized")
print(f"  proto unique values:      {df['proto'].unique() if 'proto' in df.columns else 'N/A'}")
print(f"  conn_state unique values: {df['conn_state'].unique() if 'conn_state' in df.columns else 'N/A'}")

Columns standardized
  proto unique values:      <StringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str
  conn_state unique values: <StringArray>
['S0', 'REJ', 'SF', 'RSTO', 'OTH', 'SH', 'RSTR', 'RSTRH', 'S2', 'SHR', 'S1']
Length: 11, dtype: str


## 10. Clean label_binary


In [15]:
if 'label_binary' in df.columns:
    print("label_binary before cleaning:")
    print(df['label_binary'].value_counts(dropna=False))

    # Normalize ค่าทั้งหมดให้เป็น bool
    def normalize_label_binary(val):
        if pd.isna(val):
            return False  # ถือว่าเป็น benign ถ้าไม่มี label
        s = str(val).strip().lower()
        if s in ('true', '1', 'yes', 'attack', 'malicious'):
            return True
        if s in ('false', '0', 'no', 'benign', 'normal'):
            return False
        # ค่าที่ไม่รู้จัก เช่น 'Duplicate' → log แล้วกำหนดเป็น False
        logger.warning(f"Unknown label_binary value: '{val}' → mapped to False")
        return False

    df['label_binary'] = df['label_binary'].apply(normalize_label_binary)

    print("\nlabel_binary after cleaning:")
    print(df['label_binary'].value_counts())

2026-03-30 22:13:54,951 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,952 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,952 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,953 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,953 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,953 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,954 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,954 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,954 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,955 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:54,955 - WARNING - Unknown label_binary val

label_binary before cleaning:
label_binary
True         151004
False        141049
Duplicate       607
True            243
Name: count, dtype: int64


2026-03-30 22:13:55,049 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,050 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,050 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,050 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,051 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,051 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,051 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,052 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,052 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,052 - WARNING - Unknown label_binary value: 'Duplicate' → mapped to False
2026-03-30 22:13:55,053 - WARNING - Unknown label_binary val


label_binary after cleaning:
label_binary
True     151247
False    141656
Name: count, dtype: int64


## 11. Drop Useless Columns


In [16]:
# ตรวจหาคอลัมน์ที่มีค่าเดียวตลอด (constant columns)
constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
print(f"Constant columns found ({len(constant_cols)}): {constant_cols}")

if constant_cols:
    df = df.drop(columns=constant_cols)
    print(f"Dropped {len(constant_cols)} constant columns")

# ตรวจหาคอลัมน์ที่ duplicate กับ _zeek columns
# (dest_ip, src_ip, dest_port, src_port อาจซ้ำกับ dest_ip_zeek, src_ip_zeek ฯลฯ)
redundant_check = ['dest_ip', 'src_ip', 'dest_port', 'src_port']
for col in redundant_check:
    if col in df.columns:
        zeek_col = col + '_zeek'
        if zeek_col in df.columns:
            match_rate = (df[col] == df[zeek_col]).mean()
            print(f"  {col} vs {zeek_col}: {match_rate:.2%} match")

print(f"\nFinal dataset shape: {df.shape}")

Constant columns found (6): ['protocol', 'dest_ip', 'dest_port', 'src_port', 'src_ip', 'mitre_attack_tactics']
Dropped 6 constant columns

Final dataset shape: (292903, 27)


## 12. Data Quality Checks

In [17]:
print("===== DATA CLEANING STATISTICS =====")
print(f"Initial dataset shape: {initial_shape}")
print(f"Final dataset shape:   {df.shape}")
print(f"Rows removed:          {initial_shape[0] - df.shape[0]:,}")
print(f"Rows retained:         {df.shape[0]:,}")
print(f"Retention rate:        {(df.shape[0] / initial_shape[0] * 100):.2f}%")
print(f"Columns retained:      {len(df.columns)} / {initial_shape[1]}")
print(f"\nRemaining missing values: {df.isnull().sum().sum()}")

===== DATA CLEANING STATISTICS =====
Initial dataset shape: (2385493, 33)
Final dataset shape:   (292903, 27)
Rows removed:          2,092,590
Rows retained:         292,903
Retention rate:        12.28%
Columns retained:      27 / 33

Remaining missing values: 0


## 13. Label Distribution

In [18]:
if 'label_binary' in df.columns:
    print("Label binary distribution:")
    print(df['label_binary'].value_counts())
    attack_pct = df['label_binary'].mean() * 100
    print(f"Attack ratio: {attack_pct:.2f}%")
    print()

if 'label_tactic' in df.columns:
    print("Top 10 Label Tactics:")
    print(df['label_tactic'].value_counts().head(10))
    print()

if 'proto' in df.columns:
    print("Protocols:")
    print(df['proto'].value_counts())

Label binary distribution:
label_binary
True     151247
False    141656
Name: count, dtype: int64
Attack ratio: 51.64%

Top 10 Label Tactics:
label_tactic
none                    141049
Discovery                90500
Reconnaissance           59802
Initial Access             597
Defense Evasion            587
Privilege Escalation       291
Lateral Movement            49
Exfiltration                23
Command and Control          3
Execution                    2
Name: count, dtype: int64

Protocols:
proto
tcp     155504
udp     137077
icmp       322
Name: count, dtype: int64


## 14. Statistical Summary

In [19]:
print("Statistical summary of numeric columns:")
display(df.describe())

if 'datetime' in df.columns:
    print(f"\nDate Range: {df['datetime'].min()} to {df['datetime'].max()}")
    print(f"Days covered: {(df['datetime'].max() - df['datetime'].min()).days}")

Statistical summary of numeric columns:


,duration,src_port_zeek,dest_port_zeek,missed_bytes,orig_bytes,orig_ip_bytes,orig_pkts,resp_bytes,resp_ip_bytes,resp_pkts,ts,vlan
count,292903.000000,292903.000000,292903.000000,292903.000000,2.929030e+05,292903.000000,292903.000000,2.929030e+05,2.929030e+05,292903.000000,2.929030e+05,292903.000000
mean,0.699033,43219.443430,12998.434465,11.950216,3.833556e+04,281.706183,2.463587,1.346877e+03,4.876468e+02,1.562968,1.733873e+09,50.598949
std,5.302724,12687.170946,19708.162607,1838.323972,6.986069e+06,2217.009268,10.709650,5.217386e+05,2.397230e+04,41.753996,2.576013e+07,52.372801
min,0.000000,3.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,1.639746e+09,0.000000
25%,0.000000,37709.000000,53.000000,0.000000,0.000000e+00,44.000000,1.000000,0.000000e+00,0.000000e+00,0.000000,1.727055e+09,0.000000
50%,0.000012,45357.000000,80.000000,0.000000,4.300000e+01,75.000000,1.000000,0.000000e+00,0.000000e+00,0.000000,1.748023e+09,3.000000
75%,0.000913,50907.500000,23425.000000,0.000000,1.060000e+02,162.000000,2.000000,0.000000e+00,4.000000e+01,1.000000,1.748623e+09,104.000000
max,790.689883,65535.000000,65533.000000,959379.000000,2.013427e+09,649105.000000,3845.000000,2.821382e+08,9.730316e+06,17203.000000,1.753974e+09,111.000000



Date Range: 2021-12-17 13:01:17.881000+00:00 to 2025-07-31 14:56:10.002000+00:00
Days covered: 1322


## 15. Save Cleaned Dataset

In [20]:
output_path = r'E:\thesis\dataset\UWF-ZeekData24_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
print(f"Final dataset shape: {df.shape}")

file_size = os.path.getsize(output_path) / (1024**2)
print(f"File size: {file_size:.2f} MB")

Cleaned dataset saved to: E:\thesis\dataset\UWF-ZeekData24_cleaned.csv
Final dataset shape: (292903, 27)
File size: 65.48 MB
